<a href="https://colab.research.google.com/github/rajlaxmiac/LogicMojo-AI-ML-April26-RajlaxmiSingh/blob/My-learning-Journey/Moderator_Hours_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Moderator Payable-Hours Calculator & Validator

Cross-checks the **Daily Tracker** against the **Scheduler / OData export** and computes payable hours per moderator per day.

**How to use:** run each cell top to bottom (`Shift+Enter`).
1. Install libraries
2. Upload your two files (tracker `.xlsx` + scheduler `.csv`)
3. Load the engine + data
4. Pick a moderator and date from the dropdowns — the report renders below
5. (Optional) Export everyone to a spreadsheet and download it

To adapt to another project, edit only the **CONFIG** block in the engine cell (hours table, keywords, column names, name overrides).

## 1. Install libraries

In [1]:
!pip install -q pandas openpyxl ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 22.4 MB/s eta 0:00:00


## 2. Upload your files
Run this cell, then pick your **tracker `.xlsx`** and **scheduler `.csv`** from the file dialog. You can also skip this and mount Google Drive instead (see the last cell).

In [3]:
from google.colab import files
uploaded = files.upload()   # choose the tracker .xlsx and the scheduler .csv
print('\nUploaded:', list(uploaded.keys()))

Saving scheduler-session-report.csv to scheduler-session-report.csv

Uploaded: ['scheduler-session-report.csv']


## 3. Engine
All the logic. **Edit the CONFIG block here** to change business rules. Just run it — no need to read it.

In [4]:
from __future__ import annotations

import re
import sys
import unicodedata
from dataclasses import dataclass, field
from datetime import date, datetime
from difflib import get_close_matches
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    sys.exit("Missing dependency. Run:  pip install pandas openpyxl")


# ============================================================================
#  CONFIG  — edit this block to adapt the app to another project
# ============================================================================

# Default input files (override on the command line).
DEFAULT_TRACKER = "Kilo_Mod_s_Info_and_schedule_Tracker__1_.xlsx"
DEFAULT_TRACKER_SHEET = "Daily Reporting"
DEFAULT_SCHEDULER = "scheduler-session-report-2026-07-01.csv"
DEFAULT_MODS_INFO_SHEET = "Mods Info"  # authoritative name list (optional)

# Session-count -> payable hours. Non-linear per the Kilo timesheet table.
HOURS_TABLE = {0: 0, 1: 3, 2: 6, 3: 8, 4: 10, 5: 12}
# Each session beyond the last table entry adds this many hours.
HOURS_PER_EXTRA_SESSION = 2

# Which normalized statuses are payable, and which feed the lookup index.
PAY_STATUSES = {"completed", "noshow", "valid_cancelled"}
LOOKUP_BASIS = {"completed", "noshow", "valid_cancelled"}

# Statuses removed from every calculation.
IGNORE_STATUSES = {"called_off"}

# Where the *cancelled* count comes from for pay purposes.
#   "tracker"   -> use the moderator's logged cancels, gated by the note reason.
#                  (Recommended: the scheduler/OData rarely records a *valid* cancel —
#                   device issues usually land as 'Called Off' / 'Pending Reschedule',
#                   so trusting it would under-pay legitimate device/participant cancels.)
#   "scheduler" -> only pay cancels the scheduler explicitly marks 'Cancelled'.
CANCEL_COUNT_SOURCE = "tracker"

# A cancelled session is paid only if its note mentions one of these.
VALID_CANCEL_KEYWORDS = [
    "device", "equipment", "internet", "app ", "upload", "technical",
    "tech ", "red line", "redline", "wifi", "connection",
    "family emergency", "participant issue", "pt issue", "medical",
]

# Maps raw source status strings -> canonical keys used internally.
STATUS_ALIASES = {
    "completed": "completed",
    "complete": "completed",
    "no show": "noshow",
    "no-show": "noshow",
    "noshow": "noshow",
    "cancelled": "cancelled",
    "canceled": "cancelled",
    "called off": "called_off",
    "called-off": "called_off",
    "rescheduled": "rescheduled",
    "reschedule": "rescheduled",
    "pending reschedule": "pending",
    "scheduled": "scheduled",
    "scheduling": "scheduled",
    "to be scheduled": "scheduled",
    "disqualified": "disqualified",
    "in progress": "in_progress",
}

# Confirmed name equivalences (scheduler / tracker spelling -> canonical).
NAME_MAPPING_OVERRIDES = {
    "flor corona": "Flor S. Corona",
    "flor s. corona": "Flor S. Corona",
    "lydia.coleman": "Lydia Coleman Derby",
    "lydia coleman derby": "Lydia Coleman Derby",
    "gop chand edra": "Gopi Chand Edara",
    "gopi chand edara": "Gopi Chand Edara",
    "eswar chand": "Eswar Chand Edara",
    "eswar chand edara": "Eswar Chand Edara",
    "sravya kommu": "Sravya Kommu",
    "venkata satya akash perla": "Venkata Satya Akash Perla",
    "akash perla": "Venkata Satya Akash Perla",
}

# Source column names -> canonical names the app expects.
COLUMN_MAP = {
    "tracker": {
        "Moderator name": "moderator",
        "Date": "date",
        "Scheduled Sessions": "scheduled",
        "Completed": "completed",
        "No show": "noshow",
        "Cancelled": "cancelled",
        "Notes": "notes",
        "Location": "location",
    },
    "scheduler": {
        "Primary moderator": "moderator",
        "Session date": "date",
        "Session status": "status",
        "Participant name": "participant",
        "Session type": "session_type",
    },
}

NAME_FUZZY_CUTOFF = 0.86  # difflib similarity threshold for auto-matching names


# ============================================================================
#  NORMALIZATION HELPERS
# ============================================================================

def norm_status(raw) -> str | None:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    key = str(raw).strip().lower()
    return STATUS_ALIASES.get(key, key.replace(" ", "_") or None)


def norm_name_key(raw) -> str:
    """Normalize a name to a comparison key: lowercase, no punctuation, single spaces."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return ""
    s = unicodedata.normalize("NFKD", str(raw))
    s = s.replace(".", " ").replace("_", " ").replace("-", " ")
    s = re.sub(r"[^a-zA-Z\s]", " ", s).lower()
    return re.sub(r"\s+", " ", s).strip()


ORDINAL_RE = re.compile(r"(\d{1,2})(st|nd|rd|th)", re.IGNORECASE)


def parse_flexible_date(raw):
    """Parse messy dates: '23rd May 2026', '21th June 2026', trailing spaces, real datetimes."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    if isinstance(raw, (datetime, pd.Timestamp)):
        return pd.Timestamp(raw).date()
    if isinstance(raw, date):
        return raw
    s = str(raw).strip()
    s = ORDINAL_RE.sub(r"\1", s)  # strip ordinal suffix (and typo suffixes like 21th)
    dt = pd.to_datetime(s, errors="coerce", dayfirst=False)
    if pd.isna(dt):
        dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    return None if pd.isna(dt) else dt.date()


# ============================================================================
#  NAME CANONICALIZATION  (three-tier: override -> exact -> fuzzy -> review)
# ============================================================================

class NameResolver:
    def __init__(self, canonical_names):
        self.canonical = sorted(set(n for n in canonical_names if n and str(n).strip()))
        self.key_to_canon = {norm_name_key(n): n for n in self.canonical}
        self.unresolved: set[str] = set()

    def resolve(self, raw):
        key = norm_name_key(raw)
        if not key:
            return None, "empty"
        if key in NAME_MAPPING_OVERRIDES:  # tier 0: confirmed override
            return NAME_MAPPING_OVERRIDES[key], "override"
        if key in self.key_to_canon:       # tier 1: exact normalized match
            return self.key_to_canon[key], "exact"
        # tier 2: token-subset (e.g. "eswar chand" in "eswar chand edara")
        toks = set(key.split())
        for ck, cn in self.key_to_canon.items():
            ctoks = set(ck.split())
            if toks and (toks <= ctoks or ctoks <= toks):
                return cn, "subset"
        # tier 3: fuzzy
        m = get_close_matches(key, list(self.key_to_canon), n=1, cutoff=NAME_FUZZY_CUTOFF)
        if m:
            return self.key_to_canon[m[0]], "fuzzy"
        self.unresolved.add(str(raw))       # tier 4: needs manual review
        return str(raw).strip(), "review"


# ============================================================================
#  NOTES PARSER  (extra hours + cancel-reason validity, with confidence flag)
# ============================================================================

HOUR_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(?:hours?|hrs?|hr)\b", re.IGNORECASE)
MIN_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(?:minutes?|mins?)\b", re.IGNORECASE)
BARE_HALF_RE = re.compile(r"\bmod\s*hours?\b[^0-9]*\(?\s*(0?\.\d+)\s*\)?", re.IGNORECASE)
TIME_RANGE_RE = re.compile(r"\d{1,2}(:\d{2})?\s*[ap]m\s*[-–to]+\s*\d{1,2}(:\d{2})?\s*[ap]m", re.IGNORECASE)

# A time mention only counts as *extra pay* when one of these work cues is present.
EXTRA_WORK_CUES = ("extra", "meeting", "mod hour", "office hour", "training", "trained",
                   "shadow", "setup", "set up", "waited", "wait ", "packed", "unpack",
                   "supported", "drove", "travel", "device setup")
# Words that make a number likely NOT extra work (participant lateness, reschedules).
DISTRACTOR_WORDS = ("late", "traffic", "stuck", "reschedul", "no show", "no-show", "arrived")


def parse_notes(note):
    """
    Returns dict: extra_hours (float), confident (bool), review_reasons (list[str]).
    A numeric time mention is only auto-summed when an extra-work cue is present;
    anything else is routed to Review instead of inflating pay.
    """
    result = {"extra_hours": 0.0, "confident": True, "review_reasons": []}
    if note is None or (isinstance(note, float) and pd.isna(note)):
        return result
    text = str(note).strip()
    low = text.lower()
    if not text:
        return result
    if text.isdigit():  # e.g. a stray '31'
        result["review_reasons"].append(f"note is a bare number: '{text}'")
        result["confident"] = False
        return result

    has_number = bool(HOUR_RE.search(text) or MIN_RE.search(text))
    has_cue = any(c in low for c in EXTRA_WORK_CUES)
    has_distractor = any(d in low for d in DISTRACTOR_WORDS)

    if has_number and has_cue:
        hours = sum(float(h) for h in HOUR_RE.findall(text))
        mins = sum(float(m) for m in MIN_RE.findall(text)) / 60.0
        halves = sum(float(x) for x in BARE_HALF_RE.findall(text))
        result["extra_hours"] = round(hours + mins + halves, 2)
        n_mentions = len(HOUR_RE.findall(text)) + len(MIN_RE.findall(text))
        if has_distractor or n_mentions > 1:
            result["confident"] = False
            result["review_reasons"].append(
                f"extracted {result['extra_hours']}h but note has multiple/ambiguous time mentions -> verify")
    elif has_number and not has_cue:
        result["confident"] = False
        result["review_reasons"].append("time mentioned without clear extra-work context -> not auto-counted")
    elif TIME_RANGE_RE.search(text) or any(c in low for c in EXTRA_WORK_CUES):
        result["confident"] = False
        result["review_reasons"].append("admin/meeting/training or time range mentioned but no explicit hours")

    return result


def cancel_is_valid(note) -> bool:
    if note is None or (isinstance(note, float) and pd.isna(note)):
        return False
    low = str(note).lower()
    return any(kw in low for kw in VALID_CANCEL_KEYWORDS)


# ============================================================================
#  DATA LOADING
# ============================================================================

def _rename(df, kind):
    mapping = {k: v for k, v in COLUMN_MAP[kind].items() if k in df.columns}
    missing = [c for c in COLUMN_MAP[kind] if c not in df.columns]
    if missing:
        print(f"  ! {kind}: columns not found (using defaults where possible): {missing}")
    return df.rename(columns=mapping)


def load_tracker(path, sheet):
    p = Path(path)
    if p.suffix.lower() in (".xlsx", ".xlsm", ".xls"):
        df = pd.read_excel(p, sheet_name=sheet)
    else:
        df = pd.read_csv(p)
    df = _rename(df, "tracker")
    df["date_p"] = df["date"].apply(parse_flexible_date)
    for c in ("scheduled", "completed", "noshow", "cancelled"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    if "notes" not in df.columns:
        df["notes"] = None
    return df[df["moderator"].notna()].reset_index(drop=True)


def load_scheduler(path):
    df = pd.read_csv(path)
    df = _rename(df, "scheduler")
    df["date_p"] = df["date"].apply(parse_flexible_date)
    df["status_n"] = df["status"].apply(norm_status)
    return df[df["moderator"].notna()].reset_index(drop=True)


def load_canonical_names(tracker_path, tracker_df):
    names = set(tracker_df["moderator"].dropna())
    try:
        info = pd.read_excel(tracker_path, sheet_name=DEFAULT_MODS_INFO_SHEET)
        for col in info.columns:
            if "name" in str(col).lower():
                names |= set(info[col].dropna())
    except Exception:
        pass
    # Keep only plausible person names (drop headers/labels).
    return {n for n in names if isinstance(n, str) and len(norm_name_key(n).split()) >= 2}


# ============================================================================
#  CORE CALCULATION
# ============================================================================

def lookup_hours(count: int) -> float:
    if count in HOURS_TABLE:
        return float(HOURS_TABLE[count])
    max_k = max(HOURS_TABLE)
    if count > max_k:
        return HOURS_TABLE[max_k] + (count - max_k) * HOURS_PER_EXTRA_SESSION
    return 0.0


@dataclass
class DayReport:
    moderator: str
    day: date
    location: str = ""
    # tracker-logged
    t_scheduled: int = 0
    t_completed: int = 0
    t_noshow: int = 0
    t_cancelled: int = 0
    # scheduler (source of truth)
    s_completed: int = 0
    s_noshow: int = 0
    s_cancelled: int = 0
    s_called_off: int = 0
    s_other: int = 0
    s_present: bool = False
    # verified / payable
    valid_cancelled: int = 0
    payable_count: int = 0
    session_hours: float = 0.0
    extra_hours: float = 0.0
    total_hours: float = 0.0
    # governance
    notes: str = ""
    discrepancies: list = field(default_factory=list)
    review_flags: list = field(default_factory=list)


def build_day_report(mod, day, trow, srows) -> DayReport:
    r = DayReport(moderator=mod, day=day)

    if trow is not None:
        r.location = str(trow.get("location", "") or "")
        r.t_scheduled = int(trow.get("scheduled", 0) or 0)
        r.t_completed = int(trow.get("completed", 0) or 0)
        r.t_noshow = int(trow.get("noshow", 0) or 0)
        r.t_cancelled = int(trow.get("cancelled", 0) or 0)
        r.notes = str(trow.get("notes", "") or "")

    if srows is not None and len(srows):
        r.s_present = True
        counts = srows["status_n"].value_counts().to_dict()
        r.s_completed = int(counts.get("completed", 0))
        r.s_noshow = int(counts.get("noshow", 0))
        r.s_cancelled = int(counts.get("cancelled", 0))
        r.s_called_off = int(counts.get("called_off", 0))
        r.s_other = int(sum(v for k, v in counts.items()
                            if k not in {"completed", "noshow", "cancelled", "called_off"}))

    # Completed / no-show: scheduler is source of truth when it has records.
    v_completed = r.s_completed if r.s_present else r.t_completed
    v_noshow = r.s_noshow if r.s_present else r.t_noshow

    # Cancelled: validity (device/participant) lives only in the note, and the
    # scheduler can't express a *valid* cancel, so the count is note-gated.
    cancel_count = r.t_cancelled if CANCEL_COUNT_SOURCE == "tracker" else r.s_cancelled
    if cancel_count > 0:
        if cancel_is_valid(r.notes):
            r.valid_cancelled = cancel_count
            if r.s_present and r.s_cancelled != cancel_count:
                r.review_flags.append(
                    f"{cancel_count} valid cancel(s) paid from note, but scheduler shows "
                    f"{r.s_cancelled} 'Cancelled' -> verify")
        else:
            r.valid_cancelled = 0
            r.review_flags.append(
                f"{cancel_count} cancelled session(s) with no device/participant reason "
                f"in note -> treated unpaid")

    # Notes parsing for extra hours.
    parsed = parse_notes(r.notes)
    r.extra_hours = parsed["extra_hours"]
    for reason in parsed["review_reasons"]:
        r.review_flags.append(f"notes: {reason}")

    # Payable session count -> lookup -> total.
    basis = 0
    if "completed" in LOOKUP_BASIS:
        basis += v_completed
    if "noshow" in LOOKUP_BASIS:
        basis += v_noshow
    if "valid_cancelled" in LOOKUP_BASIS:
        basis += r.valid_cancelled
    r.payable_count = basis
    r.session_hours = lookup_hours(basis)
    r.total_hours = round(r.session_hours + r.extra_hours, 2)

    # Validation: tracker vs scheduler.
    if r.s_present:
        if r.t_completed != r.s_completed:
            r.discrepancies.append(f"Completed: tracker={r.t_completed} vs scheduler={r.s_completed}")
        if r.t_noshow != r.s_noshow:
            r.discrepancies.append(f"No-show: tracker={r.t_noshow} vs scheduler={r.s_noshow}")
        if r.t_cancelled != r.s_cancelled:
            r.discrepancies.append(f"Cancelled: tracker={r.t_cancelled} vs scheduler={r.s_cancelled}")
        if r.s_called_off:
            r.discrepancies.append(f"{r.s_called_off} 'Called Off' in scheduler (ignored in pay)")
    elif (r.t_completed + r.t_noshow + r.t_cancelled) > 0:
        r.discrepancies.append("No scheduler records for this moderator/date (unverified)")

    return r


# ============================================================================
#  ENGINE
# ============================================================================

class HoursEngine:
    def __init__(self, tracker_path, tracker_sheet, scheduler_path):
        print(f"Loading tracker    : {tracker_path} [{tracker_sheet}]")
        self.tracker = load_tracker(tracker_path, tracker_sheet)
        print(f"Loading scheduler  : {scheduler_path}")
        self.scheduler = load_scheduler(scheduler_path)

        self.resolver = NameResolver(load_canonical_names(tracker_path, self.tracker))
        self.tracker["mod_c"] = self.tracker["moderator"].apply(lambda x: self.resolver.resolve(x)[0])
        self.scheduler["mod_c"] = self.scheduler["moderator"].apply(lambda x: self.resolver.resolve(x)[0])

        self.moderators = sorted(
            set(self.tracker["mod_c"].dropna()) | set(self.scheduler["mod_c"].dropna()))

    def dates_for(self, mod):
        d = set(self.tracker.loc[self.tracker["mod_c"] == mod, "date_p"].dropna())
        d |= set(self.scheduler.loc[self.scheduler["mod_c"] == mod, "date_p"].dropna())
        return sorted(d)

    def report(self, mod, day) -> DayReport:
        tr = self.tracker[(self.tracker["mod_c"] == mod) & (self.tracker["date_p"] == day)]
        trow = tr.iloc[0].to_dict() if len(tr) else None
        sr = self.scheduler[(self.scheduler["mod_c"] == mod) & (self.scheduler["date_p"] == day)]
        return build_day_report(mod, day, trow, sr if len(sr) else None)

    def all_reports(self):
        out = []
        for mod in self.moderators:
            for day in self.dates_for(mod):
                out.append(self.report(mod, day))
        return out


# ============================================================================
#  PRESENTATION
# ============================================================================

def render(r: DayReport) -> str:
    L = []
    L.append("=" * 64)
    L.append(f"  {r.moderator}   |   {r.day}   |   {r.location}")
    L.append("=" * 64)
    L.append("  SESSION BREAKDOWN            tracker   scheduler(truth)")
    L.append(f"    Completed                 {r.t_completed:>7}   {r.s_completed:>7}")
    L.append(f"    No-show (paid)            {r.t_noshow:>7}   {r.s_noshow:>7}")
    L.append(f"    Cancelled                 {r.t_cancelled:>7}   {r.s_cancelled:>7}")
    L.append(f"      -> valid (paid)         {r.valid_cancelled:>7}")
    if r.s_called_off:
        L.append(f"    Called Off (ignored)                {r.s_called_off:>7}")
    L.append("")
    status = "OK" if not r.discrepancies else "MISMATCH"
    L.append(f"  VALIDATION: {status}")
    for d in r.discrepancies:
        L.append(f"    - {d}")
    if r.review_flags:
        L.append("  REVIEW:")
        for f in r.review_flags:
            L.append(f"    * {f}")
    if r.notes.strip():
        note = r.notes.strip().replace("\n", " ")
        L.append(f"  NOTE: {note[:180]}{'...' if len(note) > 180 else ''}")
    L.append("")
    L.append("  HOURS")
    L.append(f"    Payable sessions ({r.payable_count}) -> lookup   {r.session_hours:>6.2f} h")
    L.append(f"    Extra hours (from note)              {r.extra_hours:>6.2f} h")
    L.append(f"    {'-'*38}")
    L.append(f"    TOTAL PAYABLE                        {r.total_hours:>6.2f} h")
    L.append("=" * 64)
    return "\n".join(L)


def to_dataframe(reports):
    rows = []
    for r in reports:
        rows.append({
            "Moderator": r.moderator, "Date": r.day, "Location": r.location,
            "Completed": r.s_completed if r.s_present else r.t_completed,
            "No-show": r.s_noshow if r.s_present else r.t_noshow,
            "Cancelled": r.s_cancelled if r.s_present else r.t_cancelled,
            "Valid cancelled": r.valid_cancelled,
            "Payable sessions": r.payable_count,
            "Session hours": r.session_hours,
            "Extra hours": r.extra_hours,
            "Total hours": r.total_hours,
            "Validation": "OK" if not r.discrepancies else "MISMATCH",
            "Discrepancies": " | ".join(r.discrepancies),
            "Review flags": " | ".join(r.review_flags),
            "Note": (r.notes or "").strip().replace("\n", " "),
        })
    return pd.DataFrame(rows)


def export(reports, path):
    df = to_dataframe(reports)
    if str(path).lower().endswith((".xlsx", ".xls")):
        df.to_excel(path, index=False)
    else:
        df.to_csv(path, index=False)
    print(f"\nExported {len(df)} rows -> {path}")


# ============================================================================

print('Engine loaded. Config: hours table', HOURS_TABLE, '| cancel source:', CANCEL_COUNT_SOURCE)

Engine loaded. Config: hours table {0: 0, 1: 3, 2: 6, 3: 8, 4: 10, 5: 12} | cancel source: tracker


## 4. Load the data
Auto-detects which uploaded file is the tracker (`.xlsx`) and which is the scheduler (`.csv`). If your daily-log tab isn't named `Daily Reporting`, change `TRACKER_SHEET` below.

In [5]:
import glob

TRACKER_SHEET = "Daily Reporting"   # <- change if your tab has a different name

xlsx = sorted(glob.glob('*.xlsx'))
csv  = sorted(glob.glob('*.csv'))
assert xlsx, "No .xlsx found - upload the tracker in cell 2."
assert csv,  "No .csv found - upload the scheduler in cell 2."
tracker_path, scheduler_path = xlsx[0], csv[0]
print('Tracker  :', tracker_path)
print('Scheduler:', scheduler_path)

engine = HoursEngine(tracker_path, TRACKER_SHEET, scheduler_path)
print(f'\n{len(engine.moderators)} moderators loaded.')
if engine.resolver.unresolved:
    print('Names not auto-matched (kept as-is):', sorted(engine.resolver.unresolved))

Tracker  : Kilo Mod's Info and schedule Tracker (1).xlsx
Scheduler: scheduler-session-report.csv
Loading tracker    : Kilo Mod's Info and schedule Tracker (1).xlsx [Daily Reporting]
Loading scheduler  : scheduler-session-report.csv

19 moderators loaded.


## 5. Interactive report
Pick a moderator, then a date (or **ALL DAYS**). The report renders below and updates automatically.

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

mod_dd  = widgets.Dropdown(options=engine.moderators, description='Moderator:',
                           layout=widgets.Layout(width='420px'),
                           style={'description_width':'80px'})
date_dd = widgets.Dropdown(description='Date:', layout=widgets.Layout(width='420px'),
                           style={'description_width':'80px'})
out = widgets.Output()

def refresh_dates(*_):
    days = engine.dates_for(mod_dd.value)
    date_dd.options = ['ALL DAYS'] + [str(d) for d in days]

def show_report(*_):
    with out:
        clear_output()
        mod = mod_dd.value
        if date_dd.value == 'ALL DAYS':
            total = 0.0
            for d in engine.dates_for(mod):
                r = engine.report(mod, d)
                print(render(r)); print()
                total += r.total_hours
            print(f'>>> {mod}: {len(engine.dates_for(mod))} day(s), TOTAL = {total:.2f} payable hours')
        elif date_dd.value:
            d = datetime.strptime(date_dd.value, '%Y-%m-%d').date()
            print(render(engine.report(mod, d)))

mod_dd.observe(lambda c: (refresh_dates(), show_report()), names='value')
date_dd.observe(show_report, names='value')
refresh_dates(); show_report()
display(widgets.VBox([mod_dd, date_dd, out]))

## 6. Export everyone to a spreadsheet
Builds a report for every moderator-day and downloads it. Change the filename to `.xlsx` if you prefer Excel.

In [7]:
reports = engine.all_reports()
export(reports, 'payroll_report.csv')          # or 'payroll_report.xlsx'
grand = sum(r.total_hours for r in reports)
print(f'GRAND TOTAL across {len(reports)} moderator-days: {grand:.2f} payable hours')

from google.colab import files
files.download('payroll_report.csv')


Exported 445 rows -> payroll_report.csv
GRAND TOTAL across 445 moderator-days: 2178.83 payable hours


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Optional) Use Google Drive instead of uploading
If your files live in Drive, run this *instead of* cell 2, then set `tracker_path` / `scheduler_path` in cell 4 to the Drive paths.

```python
from google.colab import drive
drive.mount('/content/drive')
# e.g. tracker_path = '/content/drive/MyDrive/Kilo/Kilo_Mod_s_Info_and_schedule_Tracker.xlsx'
```